# Rule-Based Solver Inspection

For each category that has a rule-based solver, shows every training task with:
- **Input** — the training-pair input grid
- **Expected output** — the ground-truth output
- **Prediction** — what the solver produces (green border = correct, red = wrong)

**Cell order:** 1 → 2 → 3

In [ ]:
# ── Cell 1: Clone repo and load ARC data ────────────────────────────────────
import os, io, subprocess, urllib.request, zipfile, sys
from pathlib import Path

REPO        = 'arc-agi-solver'
GITHUB_USER = 'rodehyde'
REPO_DIR    = f'/content/{REPO}'

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', f'https://github.com/{GITHUB_USER}/{REPO}.git', REPO_DIR],
                   check=True)
else:
    result = subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'],
                            capture_output=True, text=True)
    print(result.stdout or result.stderr)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f'Working directory: {os.getcwd()}')

log = subprocess.run(['git', 'log', '--oneline', '-3'], capture_output=True, text=True)
print(log.stdout)

# Download ARC training data if needed
train_dir = Path('data/training')
if not train_dir.exists() or len(list(train_dir.glob('*.json'))) < 400:
    print('Downloading ARC-AGI training data...')
    with urllib.request.urlopen(
        'https://github.com/fchollet/ARC-AGI/archive/refs/heads/master.zip'
    ) as r:
        raw = r.read()
    Path('data').mkdir(exist_ok=True)
    train_dir.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw)) as zf:
        for m in zf.namelist():
            if 'data/training/' in m and m.endswith('.json'):
                (train_dir / Path(m).name).write_bytes(zf.read(m))
    print(f'training: {len(list(train_dir.glob(".json")))} tasks')
else:
    print(f'ARC training data present ({len(list(train_dir.glob("*.json")))} tasks)')

In [ ]:
# ── Cell 2: Imports and helpers ──────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, BoundaryNorm

from scripts.human_tree import load_task as ht_load, classify
from scripts.solvers import load_task, find_solver, verify, ALL_PRIMITIVES

ARC_COLORS = [
    '#000000','#0074D9','#FF4136','#2ECC40','#FFDC00',
    '#AAAAAA','#F012BE','#FF851B','#7FDBFF','#870C25',
]
_cmap = ListedColormap(ARC_COLORS)
_norm = BoundaryNorm(range(11), 10)

def _show_grid(ax, grid, title, border_colour=None):
    ax.imshow(np.array(grid, dtype=np.uint8), cmap=_cmap, norm=_norm,
              interpolation='nearest', vmin=0, vmax=9)
    ax.set_title(title, fontsize=8)
    ax.axis('off')
    if border_colour:
        for spine in ax.spines.values():
            spine.set_edgecolor(border_colour)
            spine.set_linewidth(4)
            spine.set_visible(True)

def show_task_with_predictions(task_id, solver_name, solve_fn):
    """Show all training pairs: input | expected output | prediction."""
    task = load_task(task_id)
    pairs = task['train']
    n = len(pairs)

    # Build predictions by treating training inputs as test inputs
    check_task = {**task, 'test': [{'input': p['input']} for p in pairs]}
    try:
        preds = solve_fn(check_task) or [None] * n
    except Exception:
        preds = [None] * n

    fig, axes = plt.subplots(n, 3, figsize=(9, n * 3.2))
    if n == 1:
        axes = [axes]

    all_correct = all(
        pred is not None and np.array_equal(pred, pair['output'])
        for pred, pair in zip(preds, pairs)
    )

    for row, (pair, pred) in enumerate(zip(pairs, preds)):
        correct = pred is not None and np.array_equal(pred, pair['output'])
        pred_grid = pred if pred is not None else np.zeros_like(pair['input'])
        border = '#2ECC40' if correct else '#FF4136'

        _show_grid(axes[row][0], pair['input'],  f'Pair {row} — input')
        _show_grid(axes[row][1], pair['output'], f'Pair {row} — expected')
        _show_grid(axes[row][2], pred_grid,
                   f'Pair {row} — {solver_name} {"✓" if correct else "✗"}',
                   border_colour=border)

    status = '✓ SOLVED' if all_correct else '✗ partial / failed'
    fig.suptitle(f'{task_id}  [{status}]', fontsize=11, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

print('Helpers loaded.')
print(f'Available primitives: {[name for name, _, _ in ALL_PRIMITIVES]}')

In [ ]:
# ── Cell 3: Inspect a category ───────────────────────────────────────────────
# Set CATEGORY to any human_tree category that has a matching solver.
# Tasks are shown in order; solved tasks get a green border on the prediction.

CATEGORY = 'FILL_REGIONS'   # ← change to any category, e.g. 'COLOUR_BY_HEIGHT'

task_ids = sorted(p.stem for p in Path('data/training').glob('*.json'))
cat_ids  = [tid for tid in task_ids if classify(ht_load(tid)) == CATEGORY]

print(f'{CATEGORY}: {len(cat_ids)} tasks')
print()

n_solved = 0
for tid in cat_ids:
    task = load_task(tid)
    solver_name, preds = find_solver(task)
    if solver_name is None:
        solver_name = 'no solver'
        # Still pick the most relevant primitive to show its attempt
        from scripts.solvers import _applies_flood_fill_enclosed, _solve_flood_fill_enclosed
        from scripts.solvers import _applies_colour_by_height, _solve_colour_by_height
        # Use first primitive whose applies filter matches, or fallback
        from scripts.solvers import task_delta
        d = task_delta(task)
        if CATEGORY == 'FILL_REGIONS':
            show_fn = _solve_flood_fill_enclosed
            show_name = 'FLOOD_FILL_ENCLOSED'
        elif CATEGORY == 'COLOUR_BY_HEIGHT':
            show_fn = _solve_colour_by_height
            show_name = 'COLOUR_BY_HEIGHT'
        else:
            print(f'  {tid}: no solver for {CATEGORY} — skipping')
            continue
    else:
        n_solved += 1
        # Find the solve function for the matched solver
        from scripts.solvers import ALL_PRIMITIVES
        show_fn   = next(fn for n, _, fn in ALL_PRIMITIVES if n == solver_name)
        show_name = solver_name

    show_task_with_predictions(tid, show_name, show_fn)

print(f'\nSummary: {n_solved} / {len(cat_ids)} tasks solved by rule-based solver')